# Bradley–Terry ranking with `choix`

This notebook demonstrates how to fit a [Bradley–Terry model](https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model) on pairwise comparison data using the [`choix`](https://github.com/lucasmaystre/choix) library.

**Conventions in `choix`:**
- Items are numbered `0, 1, …, n-1`.
- Each observation is a tuple `(winner, loser)` — the winner comes first.
- Estimated parameters represent latent item strength; higher values mean stronger items.

In [ ]:
import numpy as np
import choix

np.set_printoptions(precision=3, suppress=True)

## Example 1: Coffee preference tournament

Five drinks were compared in blind taste tests. Each row records one head-to-head result.

In [ ]:
items = ["Espresso", "Latte", "Cappuccino", "Americano", "Mocha"]
n_items = len(items)

# (winner, loser) for each comparison
data = [
    (1, 0),  # Latte beats Espresso
    (0, 4),  # Espresso beats Mocha
    (3, 1),  # Americano beats Latte
    (0, 2),  # Espresso beats Cappuccino
    (2, 4),  # Cappuccino beats Mocha
    (4, 3),  # Mocha beats Americano
]

for winner, loser in data:
    print(f"  {items[winner]:>11} > {items[loser]}")

### Fit the model

`choix.ilsr_pairwise` uses iterative Luce spectral ranking — a fast maximum-likelihood estimator for pairwise data.

In [ ]:
params = choix.ilsr_pairwise(n_items, data)
print("Latent strength parameters:")
for name, strength in zip(items, params):
    print(f"  {name:>11}: {strength:+.3f}")

### Rank items

Sort by increasing parameter value to get a ranking from worst to best.

In [ ]:
ranking = np.argsort(params)  # worst to best

print("Ranking (worst → best):")
for rank, idx in enumerate(ranking, start=1):
    print(f"  {rank}. {items[idx]}")

### Predict future match outcomes

Given fitted parameters, `choix.probabilities` returns the win probability for each item in a pair.

In [ ]:
prob_latte_wins, prob_espresso_wins = choix.probabilities([1, 0], params)
print(f"P(Latte beats Espresso)     = {prob_latte_wins:.2f}")
print(f"P(Espresso beats Latte)     = {prob_espresso_wins:.2f}")

prob_espresso_wins, prob_mocha_wins = choix.probabilities([0, 4], params)
print(f"P(Espresso beats Mocha)     = {prob_espresso_wins:.2f}")

## Example 2: Compare inference algorithms

`choix` provides several estimators for the same Bradley–Terry model. On well-connected data they should agree closely.

In [ ]:
estimators = {
    "I-LSR": choix.ilsr_pairwise,
    "MM": choix.mm_pairwise,
    "Newton-CG": lambda n, d: choix.opt_pairwise(n, d, method="Newton-CG"),
    "Rank Centrality": choix.rank_centrality,
}

for name, fn in estimators.items():
    est = fn(n_items, data)
    ll = choix.log_likelihood_pairwise(data, est)
    print(f"{name:>16}  log-likelihood = {ll:8.3f}  params = {est}")

## Example 3: Simulated data

Generate synthetic comparisons from known parameters, then recover the ranking.

In [ ]:
np.random.seed(42)
n_sim = 5

true_params = choix.generate_params(n_sim, ordered=True)
sim_data = choix.generate_pairwise(true_params, n_comparisons=200)

estimated = choix.ilsr_pairwise(n_sim, sim_data)

print("True parameters:      ", true_params)
print("Estimated parameters: ", estimated)
print("RMSE:                 ", choix.rmse(true_params, estimated))

true_ranking = np.argsort(true_params)
est_ranking = np.argsort(estimated)
print("True ranking:         ", true_ranking.tolist(), "(worst → best)")
print("Estimated ranking:    ", est_ranking.tolist(), "(worst → best)")
print("Kendall tau distance: ", choix.kendalltau_dist(true_params, estimated))

## Example 4: Sparse comparisons and regularization

When the comparison graph is not fully connected (e.g. one item always wins or always loses), the maximum-likelihood estimate may not exist. Adding a small `alpha` regularizer stabilizes inference.

In [ ]:
sparse_items = ["D", "C", "B", "A"]  # A always wins, D always loses
n_sparse = len(sparse_items)
sparse_data = [(3, 2), (2, 1), (1, 0)]  # A > B > C > D, linear chain only

print("Comparisons:")
for winner, loser in sparse_data:
    print(f"  {sparse_items[winner]} > {sparse_items[loser]}")

In [ ]:
# Without regularization this often fails to converge.
try:
    choix.ilsr_pairwise(n_sparse, sparse_data)
except RuntimeError as exc:
    print(f"Unregularized fit failed: {exc}")

# A small alpha fixes the problem.
params_sparse = choix.ilsr_pairwise(n_sparse, sparse_data, alpha=0.01)
ranking_sparse = np.argsort(params_sparse)

print("\nRegularized parameters:")
for idx in ranking_sparse:
    print(f"  {sparse_items[idx]}: {params_sparse[idx]:+.3f}")